# Agent Development Kit (ADK)

**Domain:** Agentic AI  ·  **runnable:** yes

A refresher on **ADK (Agent Development Kit)** — Google's open-source framework for building, orchestrating, evaluating, and deploying multi-agent systems, with code-first primitives that make agent development feel like ordinary software engineering.

## 1. What & Why

**ADK (Agent Development Kit)** is an open-source, **code-first** framework — open-sourced by Google at Cloud Next '25 (April 2025) — for building agents and multi-agent systems. It ships a Python package (`google-adk`) and a Java SDK, and it's the same framework Google uses internally to build the agents inside its own products (Agentspace, the Customer Engagement Suite).

**The problem it solves: gluing an agent together from scratch is repetitive plumbing.** Every agent needs the same scaffolding — a loop that calls the model, parses tool calls, executes them, feeds results back, manages conversation state, streams partial output, and (eventually) delegates to other agents. ADK gives you that scaffolding as composable primitives so you write *behavior*, not orchestration glue: define a function and it becomes a tool; declare `sub_agents` and you have a routing hierarchy; wrap a pipeline in `SequentialAgent` and the steps run in order with shared state.

**Two design stances worth remembering:**
- **Code-first, not config-first.** Agents, tools, and orchestration are plain Python/Java — versionable, testable, debuggable with normal tooling. No YAML graph DSL to learn.
- **Model- and deployment-agnostic.** Optimized for Gemini + Vertex AI, but any model works via **LiteLLM** (OpenAI, Anthropic, local). Run it on your laptop, Cloud Run, GKE, or Vertex AI Agent Engine — the agent code doesn't change.

**Reach for ADK when:**
- You're building a **multi-agent** system and want first-class primitives for delegation (`sub_agents`) and deterministic pipelines (`SequentialAgent` / `ParallelAgent` / `LoopAgent`).
- You want a **batteries-included** dev loop: `adk web` (a local UI to chat with the agent and inspect every event, tool call, and state delta), `adk run`, and `adk eval`.
- You're on or near **Google Cloud / Gemini** and want a smooth path to Vertex AI Agent Engine for managed sessions, memory, and deployment.

**Skip it when:**
- You need a **single, simple tool-calling agent** — the raw model SDK's tool-use loop may be all you need; ADK's session/runner/event machinery is overhead.
- Your stack is committed to **another framework's ecosystem** ([[langgraph]], [[crewai]], [[pydanticai]]) and you don't need ADK's orchestration or Google-Cloud integration.

## 2. Mental Model

Think of ADK as **an org chart with a dispatcher.**

- The **Runner** is the dispatcher: it receives the user's message, hands it to the root agent, and pumps the resulting stream of **Events** back to you.
- An **`LlmAgent`** is a *manager*: given the request, it reasons and decides — answer directly, call a **Tool**, or **delegate** to one of its `sub_agents`.
- **Workflow agents** (`SequentialAgent`, `ParallelAgent`, `LoopAgent`) are *fixed processes*: they orchestrate their children deterministically, no LLM in the decision — run in order, run concurrently, or repeat until done.
- **Tools** are the *workers* that actually do things (call an API, run a query, compute).
- The **Session** is the *shared whiteboard*: a `state` dict plus the event history that every agent in the turn reads from and writes to.

```
   user message
        │
        ▼
   ┌─────────┐     yields      ┌────────────────────────────┐
   │ Runner  │ ──────────────▶ │ Events: text · tool_call · │
   └────┬────┘   (event stream)│ tool_result · state_delta  │
        │ invoke               └────────────────────────────┘
        ▼
   ┌──────────────── root agent (LlmAgent) ────────────────┐
   │  reason → { answer  |  call Tool  |  transfer to … }   │
   │                          │                  │          │
   │                          ▼                  ▼          │
   │                       Tool(s)          sub_agents      │
   │                                      ┌── Sequential ──┐ │
   │                                      │ Loop / Parallel│ │
   │                                      └────────────────┘ │
   └───────────────────────────────────────────────────────┘
        ▲                                                  │
        └──────────── Session.state (shared whiteboard) ◀──┘
```

One sentence: **Agents reason, workflow agents orchestrate, tools act, the Runner drives the loop, and the Session remembers.**

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`LlmAgent`** (alias `Agent`) | The reasoning unit. You give it a `model`, an `instruction` (system prompt), and a list of `tools` and/or `sub_agents`. It decides each turn whether to answer, call a tool, or delegate. |
| **Workflow agents** | Deterministic orchestrators with **no LLM in the control flow**: `SequentialAgent` (run children in order), `ParallelAgent` (run children concurrently), `LoopAgent` (repeat children until a child escalates / `max_iterations`). Compose them with `LlmAgent`s freely. |
| **Tool** | A capability the agent can invoke. A plain typed Python function becomes a `FunctionTool` automatically — ADK reads its **signature + docstring** to build the function-calling declaration sent to the model. Tools should return a `dict` (convention: include a `"status"` key). |
| **`Runner`** | The execution engine. `runner.run(...)` / `run_async(...)` takes a user message and a session, drives the agent/tool loop, and **yields `Event`s**. You don't call the agent directly — the Runner orchestrates it. |
| **`Event`** | The atomic unit of what happened: a model text chunk, a function (tool) call, a function response, or a state change. The event stream *is* the agent's output and its audit log. |
| **`Session` / `State`** | A `Session` holds one conversation's event `history` plus a mutable `state` dict. Scope state with key prefixes: `user:` (across sessions for a user), `app:` (global), `temp:` (this turn only), unprefixed (this session). |
| **`SessionService`** | Persists sessions: `InMemorySessionService` (dev), `DatabaseSessionService` (SQL), `VertexAiSessionService` (managed). |
| **`MemoryService`** | Long-term, cross-session recall (searchable knowledge), distinct from short-term session `state`. |
| **`output_key`** | Set on an agent to auto-write its final text into `session.state[output_key]` — the canonical way one step in a pipeline hands a result to the next. |
| **Callbacks** | Hooks around the loop — `before_model_callback`, `after_model_callback`, `before_tool_callback`, `before_agent_callback`, … — for guardrails, logging, caching, or short-circuiting. |
| **`AgentTool`** | Wraps an entire agent as a *tool* so a parent can call it like a function (vs. `sub_agents`, where the parent *transfers control*). |

**Delegation, two ways.** `sub_agents` = LLM-driven *transfer* (a coordinator routes the conversation to a specialist, which takes over). `AgentTool` = a parent *calls* a child agent, gets its result back, and stays in control. Pick transfer for routing, agent-as-tool for "use this specialist and continue."

**Interop.** ADK speaks **[[model-context-protocol]]** (consume MCP tool servers, or expose ADK tools over MCP) and **[[agent-to-agent]]** (expose an ADK agent over A2A, or call remote A2A agents) — so ADK is the *build* layer, MCP the *tool* layer, A2A the *cross-agent* layer.

## 4. Setup

ADK is a normal Python package. The dev UI and eval tooling come with it.

```bash
pip install google-adk
# other model providers go through LiteLLM:
pip install "google-adk[litellm]"
```

Then, from a project folder with an agent module, the batteries-included dev loop:

```bash
adk web        # local browser UI: chat with the agent, inspect events/state/traces
adk run .      # run the agent in the terminal
adk eval . evalset.json   # run an eval set against the agent
adk api_server # expose the agent over HTTP
```

To actually run an `LlmAgent` you need model credentials — either a **Gemini API key** (`GOOGLE_API_KEY`, set `GOOGLE_GENAI_USE_VERTEXAI=FALSE`) or **Vertex AI** (`GOOGLE_GENAI_USE_VERTEXAI=TRUE` + project/region).

The first two worked examples below are **dependency-free**: they reimplement ADK's *mechanics* (automatic tool-declaration, and workflow agents over shared session state) in plain Python so the notebook runs anywhere, no install or API key. The third cell shows the **real `google-adk` code** and runs it only if the package and a key are present.

In [ ]:
# Installs are optional — examples 1 & 2 are pure stdlib and run offline.
# Uncomment to get the real framework used in example 3:
# %pip install google-adk

import importlib.util, os

adk_installed = importlib.util.find_spec("google.adk") is not None
has_key = bool(os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_GENAI_USE_VERTEXAI"))
print("google-adk installed:", adk_installed)
print("model credentials present:", has_key)

## 5. Worked Examples

### Example 1 — A Python function *is* a tool

ADK's headline ergonomic: you write a normal typed function with a docstring, and ADK turns it into a `FunctionTool` by reading its **signature + docstring** to build the function-calling *declaration* the model sees. Below we reproduce that auto-declaration step with the stdlib `inspect` + `typing`, so you can see exactly what ADK generates and sends to the LLM — and the dict-with-`status` return convention ADK encourages.

In [ ]:
import inspect, typing

# A plain, typed, documented Python function. In ADK you'd just pass this in
# `tools=[get_weather]` and the framework wraps it as a FunctionTool.
def get_weather(city: str, units: str = "celsius") -> dict:
    """Get the current weather for a city.

    Args:
        city: Name of the city, e.g. "Paris".
        units: Either "celsius" or "fahrenheit".
    """
    fake = {"paris": 17, "tokyo": 22, "denver": 9}
    c = fake.get(city.lower())
    if c is None:
        return {"status": "error", "message": f"Unknown city: {city}"}
    temp = c if units == "celsius" else round(c * 9 / 5 + 32)
    return {"status": "ok", "city": city, "units": units, "temperature": temp}


# --- What ADK does under the hood: build a function declaration ---------------
_PY_TO_JSON = {str: "string", int: "integer", float: "number", bool: "boolean",
               dict: "object", list: "array"}

def build_declaration(func) -> dict:
    """Reconstruct an ADK-style function-calling declaration from a function."""
    sig = inspect.signature(func)
    hints = typing.get_type_hints(func)
    props, required = {}, []
    for name, param in sig.parameters.items():
        json_type = _PY_TO_JSON.get(hints.get(name, str), "string")
        props[name] = {"type": json_type}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    summary = (func.__doc__ or "").strip().splitlines()[0]
    return {"name": func.__name__, "description": summary,
            "parameters": {"type": "object", "properties": props, "required": required}}


import json
declaration = build_declaration(get_weather)
print("Declaration the model receives:")
print(json.dumps(declaration, indent=2))

# Simulate the model choosing to call the tool, then ADK executing it:
print("\nModel emits tool_call -> ADK runs the function:")
print(" ", get_weather("Paris"))
print(" ", get_weather("Atlantis"))

### Example 2 — Workflow agents over shared session state

The deterministic orchestrators — `SequentialAgent` and `LoopAgent` — need **no LLM**: they just run their children in a defined order and pass data through `session.state`. The convention is `output_key`: an agent writes its result to `state[output_key]`, and the next agent reads it. Below we build minimal stand-ins for `BaseAgent`, `SequentialAgent`, and `LoopAgent` over a shared `state` dict — the same contract ADK uses — and run a *generate → refine-until-good* pipeline end to end.

In [ ]:
from dataclasses import dataclass, field

# --- A minimal session: just the shared state whiteboard ---------------------
@dataclass
class Session:
    state: dict = field(default_factory=dict)

# --- Base + workflow agents (faithful to ADK's run/yield-events contract) ----
class BaseAgent:
    def __init__(self, name): self.name = name
    def run(self, session):       # yields (agent_name, action) "events"
        raise NotImplementedError

class SequentialAgent(BaseAgent):
    def __init__(self, name, sub_agents):
        super().__init__(name); self.sub_agents = sub_agents
    def run(self, session):
        for child in self.sub_agents:
            yield from child.run(session)

class LoopAgent(BaseAgent):
    """Repeat children until one escalates (signals 'done') or max_iterations."""
    def __init__(self, name, sub_agents, max_iterations=10):
        super().__init__(name); self.sub_agents = sub_agents
        self.max_iterations = max_iterations
    def run(self, session):
        for i in range(1, self.max_iterations + 1):
            for child in self.sub_agents:
                escalate = yield from child.run(session)
                if escalate:
                    yield (self.name, f"loop stopped at iteration {i}")
                    return

# --- Two concrete worker agents writing to state[output_key] ------------------
class DraftAgent(BaseAgent):
    output_key = "draft"
    def run(self, session):
        session.state[self.output_key] = "ai"          # a too-short first draft
        yield (self.name, f"wrote {self.output_key!r} = {session.state[self.output_key]!r}")

class ExpandAgent(BaseAgent):
    """Appends a word each pass; escalates once the draft is long enough."""
    output_key = "draft"
    def run(self, session):
        words = ["agents", "orchestrate", "tools", "reliably"]
        draft = session.state[self.output_key]
        n = len(draft.split())
        draft = draft + " " + words[min(n - 1, len(words) - 1)]
        session.state[self.output_key] = draft
        yield (self.name, f"draft now {draft.split().__len__()} words: {draft!r}")
        return len(draft.split()) >= 4                  # escalate => stop the loop

# --- Compose: draft once, then loop-refine until good ------------------------
pipeline = SequentialAgent("writer", [
    DraftAgent("drafter"),
    LoopAgent("refine", [ExpandAgent("expander")], max_iterations=5),
])

session = Session()
print("Event stream from runner.run():")
for agent_name, action in pipeline.run(session):
    print(f"  [{agent_name:8}] {action}")

print("\nFinal session.state:", session.state)

### Example 3 — The real thing: `LlmAgent` + `Runner` + `Session`

This is the canonical ADK shape: an `LlmAgent` with a tool, driven by a `Runner` over an `InMemorySessionService`. It calls Gemini, so it runs **only** if `google-adk` is installed *and* a `GOOGLE_API_KEY` is set; otherwise we print the exact code you'd write. (The tool is `get_weather` from Example 1 — note how little glue there is.)

In [ ]:
SNIPPET = """
import asyncio
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

agent = LlmAgent(
    name="weather_agent",
    model="gemini-2.0-flash",
    instruction="You answer weather questions. Use the get_weather tool.",
    tools=[get_weather],          # the plain Python function from Example 1
)

session_service = InMemorySessionService()
runner = Runner(agent=agent, app_name="demo", session_service=session_service)

async def main():
    await session_service.create_session(
        app_name="demo", user_id="u1", session_id="s1")
    msg = types.Content(role="user",
                        parts=[types.Part(text="What is the weather in Paris?")])
    async for event in runner.run_async(
            user_id="u1", session_id="s1", new_message=msg):
        if event.is_final_response():
            print(event.content.parts[0].text)

asyncio.run(main())
"""

if adk_installed and os.getenv("GOOGLE_API_KEY"):
    import asyncio
    from google.adk.agents import LlmAgent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai import types

    agent = LlmAgent(
        name="weather_agent",
        model="gemini-2.0-flash",
        instruction="You answer weather questions. Use the get_weather tool.",
        tools=[get_weather],
    )
    session_service = InMemorySessionService()
    runner = Runner(agent=agent, app_name="demo", session_service=session_service)

    async def main():
        await session_service.create_session(
            app_name="demo", user_id="u1", session_id="s1")
        msg = types.Content(role="user",
                            parts=[types.Part(text="What is the weather in Paris?")])
        async for event in runner.run_async(
                user_id="u1", session_id="s1", new_message=msg):
            if event.is_final_response():
                print("Agent:", event.content.parts[0].text)

    asyncio.run(main())
else:
    print("google-adk + GOOGLE_API_KEY not both present — here is the real code:")
    print(SNIPPET)

## 6. Gotchas & Pitfalls

- **Tool quality lives in the docstring and type hints.** ADK builds the function declaration from your signature + docstring, and that text is *all the model sees*. Vague docstrings, missing type hints, or untyped `**kwargs` produce a bad declaration and the model will misuse or skip the tool. Write the docstring for the LLM.
- **Return a `dict`, not a bare string.** The convention is a structured result (commonly with a `"status"` key) so the model can branch on success/error. Returning unstructured text makes downstream reasoning brittle.
- **`sub_agents` (transfer) vs `AgentTool` (call) are different.** Transfer hands the *conversation* to the sub-agent (it takes over until it transfers back); `AgentTool` calls the child, gets a result, and the parent continues. Choosing wrong gives you either a coordinator that never regains control or a specialist that can't drive a sub-flow.
- **Workflow agents have no LLM — don't expect reasoning.** `SequentialAgent`/`ParallelAgent`/`LoopAgent` only orchestrate. If you need a decision about *which* branch to run, that's an `LlmAgent`'s job. Mixing them up leads to pipelines that can't adapt.
- **`ParallelAgent` children share one `state` — mind the races.** Concurrent branches writing the same key clobber each other. Give parallel branches **distinct `output_key`s** and merge afterward.
- **State prefixes have real semantics.** `user:` persists across a user's sessions, `app:` is global, `temp:` is dropped after the turn, unprefixed is this-session only. Putting per-user data in an unprefixed key (or secrets in `app:`) is a common, quiet bug.
- **`InMemorySessionService` forgets on restart.** It's for dev. Anything you want to survive a process restart needs `DatabaseSessionService` or `VertexAiSessionService`.
- **It's async-first.** `run_async` yields an event stream; the sync `run` wraps it. Forgetting to iterate to the **final** event (or to look for `event.is_final_response()`) means you read partial output. Don't block the event loop inside a tool — make slow tools async.
- **Gemini vs Vertex routing is env-driven.** `GOOGLE_GENAI_USE_VERTEXAI` (`TRUE`/`FALSE`) plus the right keys/project decides where calls go. A "model not found" / auth error is usually this flag, not the model name.

## 7. When to Use vs Alternatives

| Framework | Best for | Trade-offs vs ADK |
|---|---|---|
| **ADK** | Code-first multi-agent systems; deterministic pipelines + LLM routing in one toolkit; smooth path to Gemini / Vertex AI Agent Engine; great local dev/eval loop (`adk web`, `adk eval`) | Younger ecosystem; leans toward Google Cloud/Gemini for the smoothest path (other models work via LiteLLM but with less polish) |
| **[[langgraph]]** | Explicit **graph** control flow, cycles, checkpointing, human-in-the-loop with fine-grained state | Lower-level and more verbose; you wire the graph yourself. ADK's workflow agents cover common shapes with less code, but LangGraph gives more control over arbitrary topologies |
| **[[crewai]]** | Fast **role/crew** mental model (agents with roles, tasks, a process) | More opinionated/higher-level; less control over the raw event loop and state than ADK |
| **[[openai-agents-sdk]]** | Lightweight agents + handoffs in the OpenAI ecosystem | Thinner orchestration; no built-in deterministic workflow agents or an equivalent local dev UI/eval harness |
| **[[pydanticai]]** | Type-safe, **structured-output**-centric single agents | Less focused on large multi-agent orchestration; pair it *with* a framework when you need many coordinated agents |
| **Raw model SDK** (`google-genai`, `anthropic`, `openai`) | One agent, a few tools, full control, minimal deps | You hand-build the tool loop, session state, streaming, and delegation — exactly the plumbing ADK removes |

**Rule of thumb:** for a **single simple agent**, the raw SDK is fine. For a **multi-agent system** — especially mixing deterministic pipelines with LLM-driven routing, and especially on Google Cloud — ADK gives you the most primitives per line. For **arbitrary graph topologies** with custom checkpointing, LangGraph; for a **role-based crew**, CrewAI.

Related notebooks: [[langgraph]], [[crewai]], [[openai-agents-sdk]], [[pydanticai]], [[model-context-protocol]], [[agent-to-agent]], [[semantic-kernel]].

## 8. Resources

- **Official docs** — https://google.github.io/adk-docs/
- **Quickstart** — https://google.github.io/adk-docs/get-started/quickstart/
- **Python SDK (`google-adk`)** — https://github.com/google/adk-python
- **Java SDK** — https://github.com/google/adk-java
- **Samples & reference agents** — https://github.com/google/adk-samples
- **API reference** — https://google.github.io/adk-docs/api-reference/
- **Announcement (Cloud Next '25)** — https://developers.googleblog.com/en/agent-development-kit-easy-to-build-multi-agent-applications/
- **Deploy to Vertex AI Agent Engine** — https://google.github.io/adk-docs/deploy/agent-engine/

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def run_turn(agent, state):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE